In [1]:
import ast
import copy
import csv
import re

import dask.dataframe as dd
import numpy as np
import pandas as pd
import seaborn as sns
from Bio import SeqIO
from Bio.Seq import Seq
from matplotlib import pyplot as plt

In [45]:
dask_controller.shutdown()

2024-04-30 09:51:45,909 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


CommClosedError: ConnectionPool closing.

In [2]:
def align_read(querystr, cigarstr, pattern=re.compile("[0-9]{0,10}[MDI]")):
    result = pattern.finditer(cigarstr)
    cigar_seq = [(item.group(0)[-1], int(item.group(0)[:-1])) for item in result]
    #     output_str = "".join(["-" for i in range(cigar[1])])
    output_str = ""
    current_idx = 0
    for item in cigar_seq:
        if item[0] == "M":
            added_str = querystr[current_idx : current_idx + item[1]]
            output_str += added_str
            current_idx += item[1]
        elif item[0] == "D":
            added_str = "".join(["-" for i in range(item[1])])
            output_str += added_str
        elif item[0] == "I":
            current_idx += item[1]
    return output_str


def cigarsfromsam(samfilepath):
    cigars = {}
    with open(samfilepath, "r") as samfile:
        for line in samfile:
            if line[0] == "@":
                next(samfile)
            else:
                splitline = line.split("\t")
                cigars[splitline[0]] = splitline[5]
    return cigars


def strsfromfasta(fastafilepath):
    queries = SeqIO.to_dict(SeqIO.parse(fastafilepath, "fasta"))
    queries = {key: str(val.seq) for key, val in queries.items()}
    return queries


def make_seg_dict(gfafile):
    segment_dict = {}
    with open(gfafile, "r") as infile:
        for line in infile:
            if line[0] == "S":
                splitline = line.split("\t")
                segment_dict[splitline[1]] = splitline[2][:-1]
    return segment_dict


def get_ref_intervals(gfafile):
    segment_dict = {}
    current_idx = 0
    with open(gfafile, "r") as infile:
        for line in infile:
            if line[0] == "S":
                splitline = line.split("\t")
                if "OFF" not in splitline[1]:
                    refstr = splitline[2][:-1]
                    strlen = len(refstr)
                    name = splitline[1]
                    if "ON" in name:
                        name = name[:-2]
                    segment_dict[name] = tuple((current_idx, current_idx + strlen))
                    current_idx += strlen
    return segment_dict


def align_read(
    querystr, refstr, cigarstr, startpos=1, pattern=re.compile("[0-9]{0,10}[MDI]")
):
    start_pos = startpos - 1  ##comes as 1 indexed from minimap
    result = pattern.finditer(cigarstr)
    cigar_seq = [(item.group(0)[-1], int(item.group(0)[:-1])) for item in result]
    #     output_str = "".join(["-" for i in range(cigar[1])])
    output_str = ""
    if start_pos > 0:
        output_str += "".join(["-" for i in range(start_pos)])
    current_idx = 0
    for item in cigar_seq:
        if item[0] == "M":
            added_str = querystr[current_idx : current_idx + item[1]]
            output_str += added_str
            current_idx += item[1]
        elif item[0] == "D":
            added_str = "".join(["-" for i in range(item[1])])
            output_str += added_str
        elif item[0] == "I":
            current_idx += item[1]
    remaining_len = len(refstr) - len(output_str)
    if remaining_len > 0:
        output_str += "".join(["-" for i in range(remaining_len)])
    return output_str


def splitstr(instr, ref_intervals):
    strassign = {key: instr[val[0] : val[1]] for key, val in ref_intervals.items()}
    return strassign


def slow_hamming_distance(s1, s2):
    if len(s1) != len(s2):
        print(s1, s2)
        raise ValueError("Strand lengths are not equal!")
    term_list = []
    for ch1, ch2 in zip(s1, s2):
        if ch1 == "N" or ch2 == "N":
            term_list.append(False)
        else:
            term_list.append(ch1 != ch2)
    result = sum(term_list)
    return result


def get_dict_dist(dict1, dict2):
    hamming_dict = {
        key: slow_hamming_distance(dict1[key], dict2[key]) for key in dict1.keys()
    }
    return hamming_dict

In [3]:
data = pd.read_csv(
    "/home/lag36/scratch/lag36/2024-03-12_lLAG3_nanopore/output_subsample=None.tsv",
    # "/home/lag36/scratch/lag36/2024-02-12_lLAG2_Nanopore-Promethion_genome_Run_2/output_subsample=None.tsv",
    delimiter="\t",
)
ref_intervals = get_ref_intervals("./ref_lLAG2.gfa")
barcodes = set(data["barcode"].tolist())

In [4]:
bit_arr = np.array([list(item) for item in barcodes]).astype("uint8")
bit_freq = np.mean(bit_arr, axis=0)

In [ ]:
fig = plt.figure(figsize=(16, 6))
sns.barplot(x=list(range(len(bit_freq))), y=bit_freq, color="grey")
plt.xlabel("Bit Number", fontsize=20)
plt.ylabel("Percent Positive", fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
# plt.savefig(
#     "/home/lag36/gutiloluis/2024-01-26_lLAG2_Nanopore_Primordium-Reads/figure_1.png",
#     dpi=300,
#     bbox_inches="tight",
# )
plt.show()

In [6]:
import os
import shutil
import time
from time import sleep

import dask
import dask.array as da
import h5py
from dask.distributed import Client, progress
from dask_jobqueue import SLURMCluster
from IPython.core.display import HTML, display


def writedir(directory, overwrite=False):
    if overwrite:
        if os.path.exists(directory):
            shutil.rmtree(directory)
        os.makedirs(directory)
    else:
        if not os.path.exists(directory):
            os.makedirs(directory)


class tr_dask_controller:  # adapted from Charles' code
    def __init__(
        self,
        n_workers=6,
        local=True,
        queue="short",
        death_timeout=3.0,
        walltime="01:30:00",
        cores=1,
        processes=1,
        memory="6GB",
        working_directory="./",
        job_extra=[],
    ):
        self.local = local
        self.n_workers = n_workers
        self.walltime = walltime
        self.queue = queue
        self.death_timeout = death_timeout
        self.processes = processes
        self.memory = memory
        self.cores = cores
        self.working_directory = working_directory
        self.job_extra = job_extra

        writedir(working_directory, overwrite=False)

    def startdask(self):
        if self.local:
            self.daskclient = Client()
            self.daskclient.cluster.scale(self.n_workers)
        else:
            self.daskcluster = SLURMCluster(
                n_workers=self.n_workers,
                queue=self.queue,
                death_timeout=self.death_timeout,
                walltime=self.walltime,
                processes=self.processes,
                memory=self.memory,
                cores=self.cores,
                local_directory=self.working_directory,
                log_directory=self.working_directory,
                job_extra=self.job_extra,
            )
            #             self.workers = self.daskcluster.start_workers(self.n_workers)
            self.daskclient = Client(self.daskcluster)

    def shutdown(self):
        self.daskclient.restart()
        if not self.local:
            self.daskcluster.close()
        for item in os.listdir(self.working_directory):
            if "worker-" in item or "slurm-" in item or ".lock" in item:
                path = "./" + item
                if os.path.isfile(path):
                    os.remove(path)
                elif os.path.isdir(path):
                    shutil.rmtree(path)

    def printprogress(self):
        complete = len([item for item in self.futures if item.status == "finished"])
        print(str(complete) + "/" + str(len(self.futures)))

    def displaydashboard(self):
        link = self.daskcluster.dashboard_link
        display(HTML('<a href="' + link + '">Dashboard</a>'))

    def mapfovs(self, function, fov_list, retries=0):
        self.function = function
        self.retries = retries

        def mapallfovs(fov_number, function=function):
            function(fov_number)

        self.futures = {}
        for fov in fov_list:
            future = self.daskclient.submit(mapallfovs, fov, retries=retries)
            self.futures[fov] = future

    def retry_failed(self):
        self.failed_fovs = [
            fov for fov, future in self.futures.items() if future.status != "finished"
        ]
        out = self.daskclient.restart()
        self.mapfovs(self.function, self.failed_fovs, retries=self.retries)

    def retry_processing(self):
        self.proc_fovs = [
            fov for fov, future in self.futures.items() if future.status == "pending"
        ]
        out = self.daskclient.restart()
        self.mapfovs(self.function, self.proc_fovs, retries=self.retries)

/tmp/ipykernel_18834/3763112403.py:11: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import HTML, display


In [15]:
dask_controller = tr_dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=5,#10,
    memory="16GB",#"32GB",
    working_directory="/home/lag36/scratch/dask",
)

dask_controller.startdask()

/home/lag36/miniconda3/envs/nanopore/lib/python3.8/site-packages/dask_jobqueue/core.py:251: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)


In [ ]:
dask_controller.daskclient

In [22]:
# dask_bit_arr = da.from_array(bit_arr, chunks=(10000, 30)).astype("uint8")

In [17]:
dask_bit_arr = da.from_array(bit_arr, chunks=(1000, 30)).astype("uint8")

In [ ]:
dask_bit_arr

In [19]:
n_chunks = 5
step = dask_bit_arr.chunksize[0] * n_chunks

closest_match_list = []

for i in range(0, dask_bit_arr.shape[0], step):
    both_on = (dask_bit_arr[i : i + step] @ dask_bit_arr.T).astype("uint8")
    both_off = ((-dask_bit_arr + 1)[i : i + step] @ (-dask_bit_arr.T + 1)).astype(
        "uint8"
    )
    ttl_match = both_on + both_off
    hamming_dist = dask_bit_arr.shape[1] - ttl_match

    N = hamming_dist.shape[0]
    M = hamming_dist.shape[1]
    K = max(0, dask_bit_arr.shape[0] - (i + step))
    if (i > 0) and (K > 0):
        zeros_arr_left = da.zeros((N, i)).astype(bool)
        zeros_arr_right = da.zeros((N, K)).astype(bool)
        diagonal_mask = da.eye(N).astype(bool)
        padded_diagonal_mask = da.concatenate(
            [zeros_arr_left, diagonal_mask, zeros_arr_right], axis=1
        )
    elif i > 0:
        zeros_arr_left = da.zeros((N, i)).astype(bool)
        diagonal_mask = da.eye(N).astype(bool)
        padded_diagonal_mask = da.concatenate([zeros_arr_left, diagonal_mask], axis=1)
    else:
        zeros_arr_right = da.zeros((N, K)).astype(bool)
        diagonal_mask = da.eye(N).astype(bool)
        padded_diagonal_mask = da.concatenate([diagonal_mask, zeros_arr_right], axis=1)

    hamming_dist[padded_diagonal_mask] = 100
    closest_match = da.min(hamming_dist, axis=1)
    closest_match_list.append(closest_match.compute())

closest_match_list = np.concatenate(closest_match_list)

In [20]:
with open(
    "./closest_match.npy",
    "wb",
) as f:
    np.save(f, closest_match_list)

In [21]:
with open(
    "./closest_match.npy",
    "rb",
) as f:
    closest_match_list = np.load(f)

In [22]:
closest_match_dict = {
    barcode: closest_match_list[k] for k, barcode in enumerate(barcodes)
}

#### These are not the plots we want in the final draft

The final distribution should come from the bit-subtracted histograms extractable from the final pandas df

In [ ]:
plt.hist(closest_match_list, bins=range(0, 9))
plt.xlabel("Closest Hamming Distance", fontsize=20)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
# plt.savefig("./figure_2.png",dpi=300,bbox_inches="tight")
plt.show()

In [ ]:
plt.hist(closest_match_list, range=(0, 1), bins=2)
plt.xlabel("Closest Hamming Distance", fontsize=20)
plt.xticks([0.25, 0.75], [0, 1], fontsize=16)
plt.yticks(fontsize=16)
# plt.savefig("./figure_3.png",dpi=300,bbox_inches="tight")
plt.show()

### Depth Distribution

In [25]:
import pickle as pkl

import scipy as sp

with open(
    "/home/lag36/scratch/lag36/2024-03-12_lLAG3_nanopore/graph_output/barcode_counts_dict.pkl",
    # "/home/lag36/scratch/lag36/2024-02-12_lLAG2_Nanopore-Promethion_genome_Run_2/graph_output/barcode_counts_dict.pkl",
    "rb",
) as infile:
    depth_dict = pkl.load(infile)

In [26]:
depth_arr = np.array(list(depth_dict.values()))

In [27]:
data["barcode depth"] = data["barcode"].apply(lambda x: depth_dict[x])


In [28]:
data['closest_match'] = data['barcode'].apply(lambda x: closest_match_dict[x])

In [29]:
np.sum(depth_arr)

42532490

In [ ]:
fig = plt.figure(figsize=(12, 8))

threshold = 200

# under_threshold_arr = depth_arr[depth_arr < threshold]
above_threshold_arr = depth_arr# depth_arr[depth_arr >= threshold]
# plt.hist(
#     under_threshold_arr,
#     bins=200,
#     range=(0, 25000),
#     log=True,
#     color="grey",
#     label="Under Threshold",
# )

plt.hist(
    above_threshold_arr,
    bins=200,
    range=(0, 25000),
    log=True,
    color="tab:blue",
    label="Above Threshold",
)
plt.xlabel("Read Depth", fontsize=20)
plt.xticks(fontsize=20)
plt.ylabel("N Barcodes", fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=20)
plt.vlines(threshold, 0, 100000, color="red", linestyle="--")
plt.savefig("./depth_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Draw a vertical line at the threshold


In [ ]:
fig = plt.figure(figsize=(12, 8))

threshold = 70

# under_threshold_arr = depth_arr[depth_arr < threshold]
above_threshold_arr = depth_arr# depth_arr[depth_arr >= threshold]
# plt.hist(
#     under_threshold_arr,
#     bins=200,
#     range=(0, 25000),
#     log=True,
#     color="grey",
#     label="Under Threshold",
# )

plt.hist(
    above_threshold_arr,
    bins=200,
    range=(0, 1000),
    log=True,
    color="tab:blue",
    label="Above Threshold",
)
plt.xlabel("Read Depth", fontsize=20)
plt.xticks(fontsize=20)
plt.ylabel("N Barcodes", fontsize=20)
plt.yticks(fontsize=20)
plt.legend(fontsize=20)
plt.vlines(threshold, 0, 100000, color="red", linestyle="--")
plt.savefig("./depth_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Draw a vertical line at the threshold


In [ ]:
## START OF DEBUGGING

In [ ]:
_ = plt.hist(data[data["closest_match"] == 1]["barcode depth"],bins=100,range=(0,2500),density=True,alpha=0.5)
_ = plt.hist(data[data["closest_match"]>3]["barcode depth"],bins=100,range=(0,2500),density=True,alpha=0.5)

In [ ]:
# END OF DEBUGGING

In [33]:
# data_df = dd.from_pandas(data, npartitions=100).persist()
data_df = dd.from_pandas(data, npartitions=1).persist()

In [ ]:
data_df.compute()

In [ ]:
## Align consensus sequence to reference using cigar string
aligned_cons = data_df.apply(
    lambda x: align_read(
        x["consensus"], x["reference"], x["cigar"], startpos=x["alignmentstart"]
    ),
    axis=1,
).persist()
## Use GFA reference to determine intervals for each annotation
ref_intervals = get_ref_intervals("./ref_lLAG2.gfa")

In [37]:
## Split sequences based on annotated intervals
split_ref = data_df.apply(
    lambda x: splitstr(x["reference"], ref_intervals), axis=1,
).persist()
split_align = aligned_cons.apply(lambda x: splitstr(x, ref_intervals)).persist()

data_df["aligned_cons"] = aligned_cons
data_df["split_ref"] = split_ref
data_df["split_align"] = split_align

## Compute hamming distance from reference, by annotated element
hamm_ref = data_df.apply(
    lambda x: get_dict_dist(x["split_align"], x["split_ref"]), axis=1, meta=dict
).persist()

gfp_mutation = data_df["split_align"].apply(lambda x: x["Variant"][75:77], meta=str).astype('category').persist()
# Dark if TA, Bright if AT
gfp_is_bright = gfp_mutation.apply(lambda s: 0 if s == 'TA' else 1 if s == 'AT' else -1).persist()

## Assign closest match to each barcode (for costructing output df later)
closest_hamming_dist = (
    data_df["barcode"].compute().apply(lambda x: closest_match_dict[x])
)

data_df["hamm_ref"] = hamm_ref
data_df["Closest Hamming Distance"] = closest_hamming_dist
data_df["gfp_mutation"] = gfp_mutation
data_df["gfp_is_bright"] = gfp_is_bright

data_df = data_df.persist()

del hamm_ref
del closest_hamming_dist
del aligned_cons
del split_align
del split_ref
del gfp_mutation
del gfp_is_bright

data_df = data_df.compute()

/home/lag36/miniconda3/envs/nanopore/lib/python3.8/site-packages/dask/dataframe/core.py:5563: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, meta=(None, 'object'))

  warnings.warn(meta_warning(meta))
/home/lag36/miniconda3/envs/nanopore/lib/python3.8/site-packages/dask/dataframe/core.py:4005: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map or apply function that you are using.
  Before: .apply(func)
  After:  .apply(func, 

### Frequency Distributions

In [ ]:
# Make a barplot of the value_counts of gfp_mutation
# Dark if TA, Bright if AT
fig = plt.figure(figsize=(3, 2))
sns.barplot(x=Sdata_df["gfp_mutation"].value_counts().index, y=data_df["gfp_mutation"].value_counts())
print(data_df["gfp_mutation"].value_counts())

In [41]:
# remove the unnamed column
data_df = data_df.loc[:, ~data_df.columns.str.contains("^Unnamed")]
# save
data_df.to_csv(
    # "./lLAG2_final_df.tsv",
    "/home/lag36/scratch/lag36/2024-03-12_lLAG3_nanopore/lLAG3_final_df.tsv",
    sep="\t",
    index=True#False
)

In [ ]:
# import pandas as pd
# pd.read_csv("./lLAG2_final_df.tsv", sep="\t").to_csv("./lLAG2_final_df_V2.tsv", sep = "\t", index=True)

In [ ]:
data_df.loc[:,'Closest Hamming Distance'].hist(bins=range(0,9))

In [ ]:
dask_controller.shutdown()

## Below here only for gRNA stuff

In [ ]:
# ## Get promoters and UMIs
# UMI = data_df["split_align"].apply(lambda x: x["Variant"][274:294], meta=str).persist()
# sgRNA = (
#     data_df["split_align"]
#     .apply(lambda x: str(Seq(x["Variant"][175:195]).reverse_complement()), meta=str)
#     .persist()
# )
# ## Assign closest match to each barcode (for costructing output df later)
# closest_hamming_dist = (
#     data_df["barcode"].compute().apply(lambda x: closest_match_dict[x])
# )

# data_df["hamm_ref"] = hamm_ref
# data_df["UMI"] = UMI
# data_df["sgRNA"] = sgRNA
# data_df["Closest Hamming Distance"] = closest_hamming_dist
# data_df = data_df.persist()

# del hamm_ref
# del closest_hamming_dist
# del aligned_cons
# del split_align
# del split_ref

# data_df = data_df.compute()

### Frequency Distributions

In [ ]:
n_barcodes = data_df.groupby("sgRNA").size()[101:]

In [ ]:
plt.hist(n_barcodes, range=(0, 10))

In [ ]:
sgRNA_data = pd.read_csv(
    "/home/de64/group/nanopore/dev_notebooks/2023-03-19_snakemake_lDE23/2021-04-20_Essential_KO_Lib_df_coords_corrected.tsv",
    sep="\t",
)
sgRNA_data = sgRNA_data[
    (sgRNA_data["Category"] != "OnlyChr")
    & (sgRNA_data["Category"] != "NonEssentialOnlyChr")
]

sgRNA_dict = sgRNA_data["sgRNA Sequence"].to_dict()
sgRNA_inv_dict = {val: key for key, val in sgRNA_dict.items()}

valid_mask = data_df["sgRNA"].isin(sgRNA_inv_dict.keys())
valid_data_df = data_df[valid_mask]
index_map = valid_data_df["sgRNA"].apply(lambda x: sgRNA_inv_dict[x])
sgRNA_data_subset = sgRNA_data.loc[index_map.to_list()]

merged_df = valid_data_df.reset_index(drop=True).join(
    sgRNA_data_subset[
        [
            "oDEPool7_id",
            "EcoWG1_id",
            "Gene",
            "Strand",
            "Target Sequence",
            "Target Sites",
            "N Target Sites",
            "N Mismatch",
            "Category",
            "TargetID",
        ]
    ].reset_index(drop=True)
)
merged_df["barcode depth"] = merged_df["barcode"].apply(lambda x: depth_dict[x])

In [ ]:
## Storing dataframe as a csv
merged_df.to_csv(
    "/home/de64/group/nanopore/dev_notebooks/2023-03-19_snakemake_lDE23/lDE23_final_df.tsv",
    sep="\t",
)

In [ ]:
dask_controller.shutdown()